In [2]:
from pyaraucaria.obs_plan.obs_plan_parser import ObsPlanParser

In [16]:
txt = "OBJECT V496_Aql 19:08:20.77 -07:26:15.89 seq=1/V/20 \n WAIT ut=16:00:00"
# txt = "WAIT ut=16:00:00"
# txt = "WAIT sec=600"
# txt = "ZERO seq=15/Ic/0"
# txt = "DARK ZZ01 seq=10/V/300,10/Ic/200"
# txt = "DOMEFLAT AL3 seq=10/i/100"
# txt = "SKYFLAT HD23 alt=60.0 az=270.0 seq=10/r/20,10/V/30"
# txt = "SKYFLAT HD24 seq=10/g/a,10/V/a"
# txt = "FOCUS NG31 12:12:12 20:20:20"
# txt = "OBJECT HD193901 20:23:35.8 -21:22:14.0 seq=1/V/300"
# txt = "OBJECT FF_Aql 18:58:14.75 17:21:39.29 seq=5/Ic/60,5/V/70"
# txt = "OBJECT V496_Aql 19:08:20.77 -07:26:15.89 seq=1/V/20"
# txt = "OBJECT HD23 alt=60.0 az=270.0"

ob = ObsPlanParser.convert_from_string(txt)

print(ob)


{'command_name': 'SEQUENCE', 'subcommands': [{'command_name': 'OBJECT', 'args': ['V496_Aql', '19:08:20.77', '-07:26:15.89'], 'kwargs': {'seq': '1/V/20'}}, {'command_name': 'WAIT', 'kwargs': {'ut': '16:00:00'}}]}


In [ ]:
import re

RA_REGEX = r"^\d{2}:\d{2}:\d{2}(\.\d+)?$"
DEC_REGEX = r"^-?\d{2}:\d{2}:\d{2}(\.\d+)?$"
UT_REGEX = r"^\d{1,2}:\d{2}:\d{2}$"

SEQ_BASIC_REGEX = r"^\d+/[A-Za-z0-9]+/(?:\d+(\.\d+)?|a)$"
SEQ_MULTIPLIER_REGEX = r"^\d+x\((.+)\)$"

DITHER_REGEX = r"^basic/\d+/\d+(\.\d+)?$"

In [ ]:
BASE_SCHEMA = {
    "type": "object",
    "required": ["command"],
    "properties": {
        "command": {
            "type": "string",
            "enum": [
                "OBJECT", "SKYFLAT", "DOMEFLAT",
                "FOCUS", "DARK", "ZERO",
                "SNAP", "STOP", "WAIT"
            ]
        },
        "args": {"type": "object"},
        "kwargs": {"type": "object"}
    },
    "additionalProperties": False
}

In [ ]:
COMMON_KWARGS = {
    "observer": {"type": "string"},
    "uobi": {"type": "integer"},
    "test": {"enum": [0, 1]},
    "read_mod": {"type": "integer"},
    "focus_offset": {"enum": ["on", "off"]},
    "mirror_cover": {"enum": ["auto", "open", "close"]},
}

In [ ]:
OBJECT_LIKE_SCHEMA = {
    "type": "object",
    "properties": {
        "command": {"enum": ["OBJECT", "SNAP", "SKYFLAT"]},
        "args": {
            "type": "object",
            "properties": {
                "object_name": {"type": "string"},
                "ra": {"type": "string", "pattern": RA_REGEX},
                "dec": {"type": "string", "pattern": DEC_REGEX},
            },
            "additionalProperties": False
        },
        "kwargs": {
            "type": "object",
            "properties": {
                "seq": {"type": "string"},
                "az": {"type": "number", "minimum": 0, "maximum": 360},
                "alt": {"type": "number", "minimum": 0, "maximum": 90},
                "tracking": {"enum": ["on", "off"]},
                "dome_follow": {"enum": ["off"]},
                "epoch": {"type": "integer"},
                "dither": {"type": "string"},
                "pi": {"type": "string"},
                "sciprog": {"type": "string"},
                **COMMON_KWARGS
            },
            "additionalProperties": False
        }
    }
}

In [ ]:
WAIT_SCHEMA = {
    "type": "object",
    "properties": {
        "command": {"const": "WAIT"},
        "kwargs": {
            "type": "object",
            "properties": {
                "sec": {"type": "number", "minimum": 0},
                "ut": {"type": "string", "pattern": UT_REGEX},
                "sunrise": {"type": "number"},
                "sunset": {"type": "number"},
                **COMMON_KWARGS
            },
            "additionalProperties": False,
            "minProperties": 1
        }
    },
    "required": ["command", "kwargs"]
}

In [ ]:
def validate_seq(seq):
    if re.match(SEQ_BASIC_REGEX, seq):
        return True

    mult_match = re.match(SEQ_MULTIPLIER_REGEX, seq)
    if mult_match:
        inner = mult_match.group(1)
        parts = inner.split(",")
        return all(re.match(SEQ_BASIC_REGEX, p.strip()) for p in parts)

    parts = seq.split(",")
    return all(re.match(SEQ_BASIC_REGEX, p.strip()) for p in parts)

In [ ]:
def validate_dither(dither):
    if dither == "off":
        return True
    return bool(re.match(DITHER_REGEX, dither))

In [ ]:
def validate_logic(cmd):
    args = cmd.get("args", {})
    kwargs = cmd.get("kwargs", {})

    ra = args.get("ra")
    dec = args.get("dec")
    alt = kwargs.get("alt")
    az = kwargs.get("az")

    # ra i dec muszą być razem
    if (ra and not dec) or (dec and not ra):
        return "args.ra/dec"

    # alt i az muszą być razem
    if (alt and not az) or (az and not alt):
        return "kwargs.alt/az"

    # nie wolno mieszać
    if (ra or dec) and (alt or az):
        return "args.ra"

    # seq validation
    if "seq" in kwargs:
        if not validate_seq(kwargs["seq"]):
            return "kwargs.seq"

    # dither validation
    if "dither" in kwargs:
        if not validate_dither(kwargs["dither"]):
            return "kwargs.dither"

    return None

In [ ]:
from jsonschema import Draft7Validator

SCHEMA_MAP = {
    "WAIT": WAIT_SCHEMA,
    "OBJECT": OBJECT_LIKE_SCHEMA,
    "SNAP": OBJECT_LIKE_SCHEMA,
    "SKYFLAT": OBJECT_LIKE_SCHEMA,
}

def validate_command(cmd):
    command = cmd.get("command")

    if command not in SCHEMA_MAP:
        return "command"

    schema = SCHEMA_MAP[command]
    validator = Draft7Validator(schema)

    errors = sorted(validator.iter_errors(cmd), key=lambda e: e.path)

    if errors:
        error = errors[0]
        if error.path:
            return ".".join(str(p) for p in error.path)
        return "unknown"

    logic_error = validate_logic(cmd)
    if logic_error:
        return logic_error

    return None